### Version 1

In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ), "GB")
else:
    print("No GPU detected. Change the runtime type to GPU.")

PyTorch version: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [ ]:
from google.colab import drive

drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
from pathlib import Path

DRIVE_DIR = Path("/content/drive/MyDrive/Modguard-AI")

print("Folder exists:", DRIVE_DIR.exists())

if DRIVE_DIR.exists():
    print("\nFiles in Google Drive folder:")
    for file in DRIVE_DIR.iterdir():
        print(file.name)
else:
    raise FileNotFoundError(
        "Could not find Modguard-AI. Check the folder name in Google Drive."
    )

Folder exists: True

Files in Google Drive folder:
ModGuard_AI_Contextual_Training.ipynb
data.zip
ModGuard_AI_DistilBERT_Training.ipynb
contextual_test.csv
contextual_train.csv
contextual_validation.csv
train_contextual.py
contextual_01


In [ ]:
import shutil
from pathlib import Path

PROJECT_DIR = Path("/content/modguard-ai")

# Create directories
(PROJECT_DIR / "src" / "models").mkdir(
    parents=True,
    exist_ok=True
)

(PROJECT_DIR / "data" / "contextual").mkdir(
    parents=True,
    exist_ok=True
)

# Copy training script
shutil.copy2(
    DRIVE_DIR / "train_contextual.py",
    PROJECT_DIR / "src" / "models" / "train_contextual.py"
)

# Copy contextual datasets
DATA_FILES = [
    "contextual_train.csv",
    "contextual_validation.csv",
    "contextual_test.csv"
]

for filename in DATA_FILES:
    shutil.copy2(
        DRIVE_DIR / filename,
        PROJECT_DIR / "data" / "contextual" / filename
    )

print("Files copied successfully!")

print("\nProject structure:")
for path in PROJECT_DIR.rglob("*"):
    print(path.relative_to(PROJECT_DIR))

Files copied successfully!

Project structure:
src
data
models
src/models
data/contextual
models/contextual_01
models/contextual_01/best_model.pt
models/contextual_01/config.json
models/contextual_01/tokenizer
models/contextual_01/best_metrics.json
models/contextual_01/tokenizer/tokenizer.json
models/contextual_01/tokenizer/tokenizer_config.json
data/contextual/contextual_validation.csv
data/contextual/contextual_train.csv
data/contextual/contextual_test.csv
src/models/train_contextual.py


In [ ]:
!pip install -q transformers scikit-learn pandas tqdm

In [ ]:
import pandas as pd

DATA_DIR = PROJECT_DIR / "data" / "contextual"

for filename in [
    "contextual_train.csv",
    "contextual_validation.csv",
    "contextual_test.csv"
]:
    df = pd.read_csv(DATA_DIR / filename)

    print(f"\n{filename}")
    print("Rows:", len(df))
    print("Columns:", len(df.columns))


contextual_train.csv
Rows: 500
Columns: 12

contextual_validation.csv
Rows: 120
Columns: 12

contextual_test.csv
Rows: 120
Columns: 12


In [ ]:
%cd /content/modguard-ai

/content/modguard-ai


In [ ]:
from pathlib import Path

script_path = Path(
    "/content/modguard-ai/src/models/train_contextual.py"
)

source = script_path.read_text()

old_code = '''    tokens["labels"] = labels'''

new_code = '''    # DistilBERT does not use token_type_ids
    tokens.pop("token_type_ids", None)

    tokens["labels"] = labels'''

if old_code not in source:
    raise ValueError(
        "Could not find the expected code. Script was not modified."
    )

source = source.replace(old_code, new_code, 1)

script_path.write_text(source)

print("Training script successfully patched!")

# Display the modified section
lines = script_path.read_text().splitlines()

for i, line in enumerate(lines):
    if 'tokens.pop("token_type_ids"' in line:
        print("\nModified code:")
        print("\n".join(lines[i-3:i+5]))
        break

Training script successfully patched!

Modified code:
    )

    # DistilBERT does not use token_type_ids
    tokens.pop("token_type_ids", None)

    tokens["labels"] = labels

    return tokens


In [ ]:
!python src/models/train_contextual.py

Device: cuda
Training examples: 500
Validation examples: 120
Loading weights: 100% 100/100 [00:00<00:00, 17722.91it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Epoch 1/3: 100% 125/125 [00:05<00:00, 21.20it/s, loss=0.0966]

Epoch 1
Training loss: 0.3277071741819382
{
  "toxicity": {
    "precision": 0.7272727272727273,
    "recall": 0.6666666666666666,
    "f1": 0.6956521739130435,
    "average_precision": 0.8008301612618156,
    "support": 72
  },
  "severe_toxicity": {
    "precision": 0.0,
    "recall": 0.0,
    "f1": 0.0,
    "average_pr

In [ ]:
import shutil
from pathlib import Path

SOURCE = Path("/content/modguard-ai/models/contextual_01")

DESTINATION = DRIVE_DIR / "contextual_01"

if not SOURCE.exists():
    raise FileNotFoundError(
        "Training output folder not found. Check whether training completed."
    )

if DESTINATION.exists():
    shutil.rmtree(DESTINATION)

shutil.copytree(SOURCE, DESTINATION)

print("Model successfully saved to Google Drive!")

for path in DESTINATION.rglob("*"):
    print(path.relative_to(DRIVE_DIR))

Model successfully saved to Google Drive!
contextual_01/best_model.pt
contextual_01/config.json
contextual_01/tokenizer
contextual_01/best_metrics.json
contextual_01/tokenizer/tokenizer.json
contextual_01/tokenizer/tokenizer_config.json


In [ ]:
from google.colab import files
import shutil

shutil.make_archive(
    "/content/contextual_01",
    "zip",
    root_dir="/content/drive/MyDrive/Modguard-AI/contextual_01"
)

files.download("/content/contextual_01.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### Version 2

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving train_contextual_v3.py to train_contextual_v3.py


In [ ]:
!pip install -q transformers scikit-learn pandas tqdm

In [ ]:
import os
import zipfile

os.makedirs("/content/modguard-ai/src/models", exist_ok=True)

with zipfile.ZipFile("/content/contextual_v3_data.zip", "r") as z:
    z.extractall("/content/modguard-ai")

!cp /content/train_contextual_v3.py /content/modguard-ai/src/models/

%cd /content/modguard-ai

!ls data/experiments/v3/

/content/modguard-ai
dataset_report.json  test.csv  train.csv  validation.csv


In [ ]:
!python -m src.models.train_contextual_v3

Device: cuda
Train rows: 1000 | Validation rows: 250
Training label positives:
{'toxicity': 500, 'severe_toxicity': 250, 'obscene': 250, 'threat': 250, 'insult': 250, 'identity_attack': 250, 'sexual_explicit': 250}
config.json: 100% 483/483 [00:00<00:00, 1.96MB/s]
tokenizer_config.json: 100% 48.0/48.0 [00:00<00:00, 191kB/s]
vocab.txt: 100% 232k/232k [00:00<00:00, 16.7MB/s]
tokenizer.json: 100% 466k/466k [00:00<00:00, 13.7MB/s]

model.safetensors: downloading bytes:  90% 241M/268M [00:01<00:00, 227MB/s, 21.0MB/s  ]
model.safetensors: reconstructing file:  26% 68.6M/268M [00:01<00:04, 40.2MB/s]
model.safetensors: downloading bytes: 100% 250M/250M [00:04<00:00, 55.5MB/s, 23.1MB/s  ]
model.safetensors: reconstructing file: 100% 268M/268M [00:04<00:00, 59.4MB/s, 20.4MB/s  ]
Loading weights: 100% 100/100 [00:00<00:00, 6005.50it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
voca

In [ ]:
import json

with open("models/contextual_02/best_metrics.json") as f:
    print(json.dumps(json.load(f), indent=2))

{
  "toxicity": {
    "precision_at_0.5": 1.0,
    "recall_at_0.5": 1.0,
    "f1_at_0.5": 1.0,
    "average_precision": 1.0,
    "positive_support": 125
  },
  "severe_toxicity": {
    "precision_at_0.5": 1.0,
    "recall_at_0.5": 1.0,
    "f1_at_0.5": 1.0,
    "average_precision": 1.0,
    "positive_support": 62
  },
  "obscene": {
    "precision_at_0.5": 1.0,
    "recall_at_0.5": 1.0,
    "f1_at_0.5": 1.0,
    "average_precision": 1.0,
    "positive_support": 62
  },
  "threat": {
    "precision_at_0.5": 1.0,
    "recall_at_0.5": 1.0,
    "f1_at_0.5": 1.0,
    "average_precision": 1.0,
    "positive_support": 62
  },
  "insult": {
    "precision_at_0.5": 1.0,
    "recall_at_0.5": 1.0,
    "f1_at_0.5": 1.0,
    "average_precision": 1.0,
    "positive_support": 62
  },
  "identity_attack": {
    "precision_at_0.5": 1.0,
    "recall_at_0.5": 1.0,
    "f1_at_0.5": 1.0,
    "average_precision": 1.0,
    "positive_support": 62
  },
  "sexual_explicit": {
    "precision_at_0.5": 1.0,
    "r

In [ ]:
!zip -r contextual_02.zip models/contextual_02

  adding: models/contextual_02/ (stored 0%)
  adding: models/contextual_02/best_model.pt (deflated 8%)
  adding: models/contextual_02/best_metrics.json (deflated 86%)
  adding: models/contextual_02/tokenizer/ (stored 0%)
  adding: models/contextual_02/tokenizer/tokenizer_config.json (deflated 43%)
  adding: models/contextual_02/tokenizer/tokenizer.json (deflated 71%)
  adding: models/contextual_02/config.json (deflated 45%)


In [ ]:
from google.colab import files
files.download("contextual_02.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>